# BRFSS 2025 Diabetes Classification
## Data Understanding and Literature-Grounded Variable Selection

**Study objective:** compare machine-learning algorithms for classifying **self-reported diagnosed diabetes status** using demographic, behavioral, socioeconomic, access-to-care, and health indicators from the 2025 Behavioral Risk Factor Surveillance System (BRFSS).

This notebook covers **Part I only**: dataset rationale, initial inspection, target inspection, and literature-grounded predictor selection.

> No preprocessing pipeline, train/test split, resampling, hyperparameter tuning, or model fitting is performed here.



# I. BRFSS 2025 Dataset and Prior Research

## 1.1 Why BRFSS for Diabetes Classification?

The Behavioral Risk Factor Surveillance System (BRFSS) is a large U.S. health survey coordinated by the Centers for Disease Control and Prevention (CDC). It collects information on health conditions, health-related behaviors, healthcare access, and demographic characteristics.

BRFSS 2025 is suitable for this study because it provides:

- a large population-level sample;
- a diabetes-status question (`DIABETE4`);
- demographic, behavioral, socioeconomic, and chronic-condition indicators in the same survey;
- publicly documented variable definitions and coding;
- a recent 2025 snapshot for an up-to-date comparative ML study.

### Important limitation

The diabetes outcome is **self-reported diagnosed diabetes status**. It is not a laboratory diagnosis, not a screen for undiagnosed diabetes, and not a longitudinal prediction of future diabetes.

**Official data source:** CDC BRFSS 2025 annual data.



## 1.2 Prior BRFSS Diabetes Machine-Learning Research

Our design is grounded in prior BRFSS-based diabetes studies rather than selecting variables or algorithms only by intuition.

Key anchors include:

1. **Nayem & Biswas (2026), Scientific Reports** — raw BRFSS 2014–2024, large-scale logistic modeling, literature-grounded predictor selection, public R code. DOI: `10.1038/s41598-026-46927-7`.
2. **Xie et al. (2019), Preventing Chronic Disease / CDC** — raw BRFSS 2014, literature-derived diabetes predictors, multiple ML models, leakage-safe train-only SMOTE. DOI: `10.5888/pcd16.190109`.
3. **Muhammad et al. (2025), Journal of Primary Care & Community Health** — BRFSS 2023 diabetes classification with seven ML algorithms, stratified split, 5-fold CV, imbalance-aware metrics, and SHAP. DOI: `10.1177/21501319251400546`.
4. **Aich et al. (2026), Scientific Reports** — BRFSS-derived diabetes health indicators with public Python code and reproducible feature-selection experiments. DOI: `10.1038/s41598-026-41874-9`.

These studies provide precedents for target construction, candidate-variable domains, evaluation design, feature selection, and class-imbalance handling.

**Reproducibility note:** literature is treated as evidence, not as unquestioned ground truth. Where published text and public code disagree, the discrepancy is documented and independently checked.



# II. Initial Exploration of BRFSS 2025

## 2.1 Data Source and File Integrity

The source file used here is the official CDC 2025 SAS Transport dataset (`.XPT`). The GitHub/Kaggle pipeline verifies the payload before analysis.

The original CDC ZIP is preserved separately. Kaggle uses a normalized archive member name because the CDC ZIP contains an XPT filename with a trailing space.



In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import pandas as pd
from IPython.display import display

TARGET = "DIABETE4"
EXPECTED_XPT_BYTES = 799_971_280

xpt_candidates = list(Path("/kaggle/input").rglob("LLCP2025.XPT"))
zip_candidates = list(Path("/kaggle/input").rglob("LLCP2025XPT.zip"))

print("XPT candidates:", [str(p) for p in xpt_candidates])
print("ZIP candidates:", [str(p) for p in zip_candidates])

if len(xpt_candidates) == 1:
    xpt_path = xpt_candidates[0]
    source_mode = "Kaggle dataset extracted XPT"
elif len(xpt_candidates) == 0 and len(zip_candidates) == 1:
    zip_path = zip_candidates[0]
    working_dir = Path("/kaggle/working/brfss_2025")
    working_dir.mkdir(parents=True, exist_ok=True)

    with ZipFile(zip_path) as zf:
        members = [info for info in zf.infolist() if not info.is_dir()]
        if len(members) != 1:
            raise RuntimeError(f"Expected one data payload, found {len(members)}.")
        member = members[0]
        normalized_name = Path(member.filename.strip()).name
        if normalized_name != "LLCP2025.XPT":
            raise RuntimeError(f"Unexpected payload name: {normalized_name!r}")
        xpt_path = working_dir / normalized_name
        with zf.open(member) as src, xpt_path.open("wb") as dst:
            for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                dst.write(chunk)
    source_mode = "Kaggle dataset ZIP fallback"
else:
    raise RuntimeError(
        "Expected exactly one BRFSS input payload from Kaggle Dataset. "
        f"Found {len(xpt_candidates)} XPT and {len(zip_candidates)} ZIP candidates."
    )

actual_bytes = xpt_path.stat().st_size
print("Source mode:", source_mode)
print("XPT path:", xpt_path)
print("XPT bytes:", f"{actual_bytes:,}")

if actual_bytes != EXPECTED_XPT_BYTES:
    raise RuntimeError(
        f"Unexpected XPT size: {actual_bytes:,}; expected {EXPECTED_XPT_BYTES:,}."
    )

sha256 = hashlib.sha256()
with xpt_path.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)
xpt_sha256 = sha256.hexdigest()
print("XPT SHA-256:", xpt_sha256)


In [ ]:
print("Validated Kaggle input payload.")
print("Ready to load SAS Transport file:", xpt_path)


## 2.2 Load and Inspect the Dataset

We first verify the shape, memory footprint, target availability, and a small sample of rows before performing any analytical recoding.



In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

print("Shape:", df.shape)
print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])
print("Memory usage:", f"{df.memory_usage(deep=True).sum() / (1024**3):.3f} GiB")

if TARGET not in df.columns:
    raise RuntimeError(f"Target {TARGET!r} not found in dataset.")

display(df.head())


## 2.3 Target Variable — `DIABETE4`

`DIABETE4` is the candidate outcome variable. We inspect its raw response distribution **before** recoding.

Primary study definition, frozen after literature and coding review:

- `1` → Diabetes
- `3` → No diabetes
- `2` → exclude (pregnancy-only diabetes)
- `4` → exclude (prediabetes / borderline)
- `7` → exclude (don't know / not sure)
- `9` → exclude (refused)
- missing → exclude

This creates a conservative binary comparison of diagnosed diabetes versus no diagnosed diabetes.



In [ ]:
target_counts = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis(TARGET)
    .reset_index(name="count")
)
target_counts["percent"] = target_counts["count"] / len(df) * 100

display(target_counts)
target_counts.to_csv("/kaggle/working/target_distribution.csv", index=False)

primary_target = df[TARGET].isin([1.0, 3.0])
n_primary = int(primary_target.sum())
n_diabetes = int((df[TARGET] == 1.0).sum())
n_no_diabetes = int((df[TARGET] == 3.0).sum())

print(f"Primary binary cohort: {n_primary:,}")
print(f"Diabetes: {n_diabetes:,}")
print(f"No diabetes: {n_no_diabetes:,}")
print(f"Diabetes prevalence in binary cohort: {n_diabetes / n_primary * 100:.2f}%")


## 2.4 Missingness and Variable Coverage

BRFSS uses two kinds of missingness:

1. native SAS missing values, which appear as `NaN`;
2. explicit response codes such as `7`, `9`, `77`, or `99` for don't know/refused/not sure, depending on the variable.

Therefore, raw `NaN` percentages are useful for inspection but are **not yet analytical missingness**. Coding-specific missing values are handled in the variable coding audit before preprocessing.



In [ ]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100),
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
}).sort_values(["missing_pct", "column"], ascending=[False, True])

print("Columns with the highest native SAS missingness:")
display(column_summary.head(30))

column_summary.to_csv("/kaggle/working/column_summary.csv", index=False)


# III. Literature-Grounded Variable Selection

## 3.1 Selection Criteria

A variable is retained only when it satisfies the study design rather than simply showing a strong univariate association with the target.

The selection criteria are:

1. **Literature support** — previously used or justified in BRFSS diabetes research.
2. **Analytical coverage** — acceptable missingness after decoding BRFSS special values.
3. **No target leakage** — the variable must not reveal treatment, monitoring, or history conditional on an existing diabetes diagnosis.
4. **Limited redundancy** — avoid duplicate representations of the same construct.
5. **Interpretability and domain coverage** — retain a compact set covering major health and social domains.



## 3.2 Explicit Target-Leakage Exclusions

The following variables are **not permitted** as predictors:

- `DIABAGE4` — age at diabetes diagnosis;
- `INSULIN1` — insulin use;
- `CHKHEMO3` — diabetes-specific HbA1c monitoring;
- `EYEEXAM1` — diabetes-specific eye examination;
- `DIABEYE1` — diabetes-related eye disease;
- `DIABEDU1` — diabetes self-management education.

These variables are downstream of, or conditional on, a known diabetes diagnosis. Their exclusion is based on leakage risk, not merely their high missingness.



## 3.3 Final Primary Predictor Set

The primary predictor set is frozen at **20 variables**.

It intentionally covers demographics, socioeconomic status, anthropometrics, general/mental health, lifestyle, cardiometabolic health, major comorbidities, functional health, and healthcare access while avoiding unnecessary duplication.



In [ ]:
PRIMARY_FEATURES = [
    "_AGEG5YR",
    "_SEX",
    "_RACEGR3",
    "_EDUCAG",
    "_INCOMG1",
    "EMPLOY1",
    "_BMI5",
    "GENHLTH",
    "PHYSHLTH",
    "MENTHLTH",
    "_TOTINDA",
    "_RFSMOK3",
    "SLEPTIM1",
    "_RFHYPE6",
    "_RFCHOL3",
    "_MICHD",
    "CVDSTRK3",
    "CHCKDNY2",
    "DIFFWALK",
    "PERSDOC3"
]

feature_domains = {
    "_AGEG5YR": "Demographic",
    "_SEX": "Demographic",
    "_RACEGR3": "Demographic",
    "_EDUCAG": "Socioeconomic",
    "_INCOMG1": "Socioeconomic",
    "EMPLOY1": "Socioeconomic",
    "_BMI5": "Anthropometric",
    "GENHLTH": "General health",
    "PHYSHLTH": "General health",
    "MENTHLTH": "Mental health",
    "_TOTINDA": "Lifestyle",
    "_RFSMOK3": "Lifestyle",
    "SLEPTIM1": "Lifestyle",
    "_RFHYPE6": "Cardiometabolic",
    "_RFCHOL3": "Cardiometabolic",
    "_MICHD": "Comorbidity",
    "CVDSTRK3": "Comorbidity",
    "CHCKDNY2": "Comorbidity",
    "DIFFWALK": "Functional health",
    "PERSDOC3": "Healthcare access",
}

missing_features = [c for c in PRIMARY_FEATURES if c not in df.columns]
if missing_features:
    raise RuntimeError(f"Frozen predictors missing from BRFSS 2025: {missing_features}")

feature_audit = pd.DataFrame({
    "variable": PRIMARY_FEATURES,
    "domain": [feature_domains[c] for c in PRIMARY_FEATURES],
    "native_missing_pct": [df[c].isna().mean() * 100 for c in PRIMARY_FEATURES],
    "n_unique_nonnull": [df[c].nunique(dropna=True) for c in PRIMARY_FEATURES],
})

display(feature_audit)
feature_audit.to_csv("/kaggle/working/final_feature_audit.csv", index=False)


## 3.4 Why Some Plausible Variables Were Not Selected

Examples of deliberate exclusions:

- `_BMI5CAT` — redundant with continuous `_BMI5`.
- `_RFHLTH` — derived from `GENHLTH`.
- `MARITAL` — lower priority after age, sex, education, income, employment, and race/ethnicity are represented.
- `_HLTHPL2`, `MEDCOST1`, `CHECKUP1` — useful access/use variables, but only `PERSDOC3` is retained in the compact primary set.
- `DRNKANY6`, `_RFBING6` — valid lifestyle variables, but activity, smoking, and sleep already provide interpretable behavioral coverage.
- `ASTHMA3`, `HAVARTH4` — weaker diabetes-specific rationale than the selected cardiovascular, kidney, and functional-health indicators.
- `ADDEPEV3` — overlaps the broader `MENTHLTH` construct.

These variables remain available for sensitivity analyses but are not part of the primary benchmark.



## 3.5 Interpretation Note

`_RACEGR3` is retained as a population-demographic covariate because prior raw-BRFSS studies include race/ethnicity and because diabetes prevalence can differ across population groups.

Any association must **not** be interpreted as a biological causal effect. Observed differences may reflect social, environmental, healthcare-access, socioeconomic, and structural factors not fully represented by the remaining variables.



# IV. Part I Summary

At the end of Part I we have:

- verified and loaded the official BRFSS 2025 public-use file;
- inspected the raw dataset structure and missingness;
- defined a conservative binary `DIABETE4` target;
- grounded predictor selection in prior BRFSS diabetes literature;
- screened diabetes-specific target leakage;
- frozen a compact **20-predictor** primary feature set.

The next notebook/section will address **analytical recoding, missing-data handling, encoding, train/test separation, and cross-validation**. None of those transformations should be fit using information from the final test set.



In [ ]:
overview = {
    "source_mode": source_mode,
    "source_file": xpt_path.name,
    "xpt_sha256": xpt_sha256,
    "xpt_bytes": int(actual_bytes),
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "target": TARGET,
    "target_missing": int(df[TARGET].isna().sum()),
    "memory_gib": round(float(df.memory_usage(deep=True).sum() / (1024**3)), 3),
}

with open("/kaggle/working/dataset_overview.json", "w", encoding="utf-8") as f:
    json.dump(overview, f, indent=2)

print(json.dumps(overview, indent=2))
print("Initial exploration complete.")


# Part II — Data Preparation

## 2.1 Preprocessing Principles

Part II converts the raw BRFSS responses into a model-ready dataset while preventing data leakage.

The rules are:

1. construct the binary target first and exclude non-target response categories;
2. decode BRFSS special values such as `7`, `9`, `77`, and `99` before measuring analytical missingness;
3. split train/test **before fitting any imputer, encoder, or scaler**;
4. fit preprocessing only on the training set;
5. keep the test set untouched until final model evaluation;
6. perform 5-fold cross-validation only within the training set.

This project is a **machine-learning classification comparison**, not a design-based BRFSS prevalence analysis. Survey weights such as `_LLCPWT` are therefore not used as predictors or for the primary model benchmark. Any population prevalence estimate would require the BRFSS complex-sampling design and weights.



## 2.2 Construct the Binary Target and Recode the 20 Predictors

The primary outcome compares respondents reporting diagnosed diabetes (`DIABETE4=1`) with respondents reporting no diabetes (`DIABETE4=3`).

Gestational diabetes, prediabetes/borderline diabetes, don't-know/refused responses, and missing target responses are excluded from the primary binary cohort.

For predictors, CDC non-response codes are converted to analytical missing values. Examples:

- `_AGEG5YR=14` → missing;
- `_EDUCAG=9` → missing;
- `_INCOMG1=9` → missing;
- `PHYSHLTH/MENTHLTH=88` → zero days, while 77/99 → missing;
- `SLEPTIM1=1..24` → valid hours, while 77/99 → missing;
- calculated binary variables with code 9 → missing.

The continuous BMI field `_BMI5` is stored as BMI × 100 and is divided by 100.



In [ ]:
import numpy as np

PRIMARY_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "_BMI5", "GENHLTH", "PHYSHLTH", "MENTHLTH", "_TOTINDA", "_RFSMOK3",
    "SLEPTIM1", "_RFHYPE6", "_RFCHOL3", "_MICHD", "CVDSTRK3",
    "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

CATEGORICAL_FEATURES = [
    "_AGEG5YR", "_SEX", "_RACEGR3", "_EDUCAG", "_INCOMG1", "EMPLOY1",
    "GENHLTH", "_TOTINDA", "_RFSMOK3", "_RFHYPE6", "_RFCHOL3", "_MICHD",
    "CVDSTRK3", "CHCKDNY2", "DIFFWALK", "PERSDOC3",
]

NUMERIC_FEATURES = ["_BMI5", "PHYSHLTH", "MENTHLTH", "SLEPTIM1"]

def keep_codes(series, valid_codes):
    return series.where(series.isin(valid_codes), np.nan)

def recode_brfss_2025(raw):
    out = pd.DataFrame(index=raw.index)

    # Categorical / ordinal survey variables
    out["_AGEG5YR"] = keep_codes(raw["_AGEG5YR"], range(1, 14))
    out["_SEX"] = keep_codes(raw["_SEX"], [1, 2])
    out["_RACEGR3"] = keep_codes(raw["_RACEGR3"], range(1, 6))
    out["_EDUCAG"] = keep_codes(raw["_EDUCAG"], range(1, 5))
    out["_INCOMG1"] = keep_codes(raw["_INCOMG1"], range(1, 8))
    out["EMPLOY1"] = keep_codes(raw["EMPLOY1"], range(1, 9))
    out["GENHLTH"] = keep_codes(raw["GENHLTH"], range(1, 6))
    out["_TOTINDA"] = keep_codes(raw["_TOTINDA"], [1, 2])
    out["_RFSMOK3"] = keep_codes(raw["_RFSMOK3"], [1, 2])
    out["_RFHYPE6"] = keep_codes(raw["_RFHYPE6"], [1, 2])
    out["_RFCHOL3"] = keep_codes(raw["_RFCHOL3"], [1, 2])
    out["_MICHD"] = keep_codes(raw["_MICHD"], [1, 2])
    out["CVDSTRK3"] = keep_codes(raw["CVDSTRK3"], [1, 2])
    out["CHCKDNY2"] = keep_codes(raw["CHCKDNY2"], [1, 2])
    out["DIFFWALK"] = keep_codes(raw["DIFFWALK"], [1, 2])
    out["PERSDOC3"] = keep_codes(raw["PERSDOC3"], [1, 2, 3])

    # Continuous variables
    out["_BMI5"] = raw["_BMI5"] / 100.0

    for col in ["PHYSHLTH", "MENTHLTH"]:
        s = raw[col].replace({88.0: 0.0, 77.0: np.nan, 99.0: np.nan})
        out[col] = s.where(s.between(0, 30), np.nan)

    sleep = raw["SLEPTIM1"].replace({77.0: np.nan, 99.0: np.nan})
    out["SLEPTIM1"] = sleep.where(sleep.between(1, 24), np.nan)

    return out[PRIMARY_FEATURES]

target_mask = df["DIABETE4"].isin([1.0, 3.0])
analysis_raw = df.loc[target_mask, PRIMARY_FEATURES].copy()
X = recode_brfss_2025(analysis_raw)
y = (df.loc[target_mask, "DIABETE4"] == 1.0).astype("int8").rename("diabetes")

assert len(X) == len(y)
assert X.columns.tolist() == PRIMARY_FEATURES
assert set(y.unique()) == {0, 1}

print("Analysis cohort:", f"{len(X):,}")
print("Diabetes:", f"{int(y.sum()):,}")
print("No diabetes:", f"{int((y == 0).sum()):,}")
print("Sample diabetes prevalence:", f"{y.mean() * 100:.2f}%")
display(X.head())


## 2.3 Analytical Missingness After Recoding

Analytical missingness is recomputed **after** special-value decoding. This is the missingness relevant to preprocessing.

We also quantify how many respondents would remain under a strict complete-case strategy. This provides an empirical reason for choosing imputation rather than automatically discarding every row with at least one missing predictor.



In [ ]:
analytical_missingness = pd.DataFrame({
    "variable": PRIMARY_FEATURES,
    "domain": [feature_domains[c] for c in PRIMARY_FEATURES],
    "missing_count": [int(X[c].isna().sum()) for c in PRIMARY_FEATURES],
    "missing_pct": [float(X[c].isna().mean() * 100) for c in PRIMARY_FEATURES],
    "n_unique_nonnull": [int(X[c].nunique(dropna=True)) for c in PRIMARY_FEATURES],
}).sort_values("missing_pct", ascending=False)

complete_case_mask = X.notna().all(axis=1)
complete_case_n = int(complete_case_mask.sum())
complete_case_pct = complete_case_n / len(X) * 100

print(f"Rows in primary cohort: {len(X):,}")
print(f"Complete cases across all 20 predictors: {complete_case_n:,} ({complete_case_pct:.2f}%)")
print(f"Rows lost by complete-case deletion: {len(X)-complete_case_n:,} ({100-complete_case_pct:.2f}%)")

display(analytical_missingness)
analytical_missingness.to_csv("/kaggle/working/analytical_missingness.csv", index=False)


## 2.4 Missing-Data Strategy

The primary pipeline **retains the full binary target cohort** and handles missing predictors inside the training pipeline:

- **Categorical predictors:** missing values become an explicit `-1` category, then one-hot encoded.
- **Continuous predictors:** missing values are median-imputed using the training data only.
- **Continuous scaling:** standardized using training-set mean and standard deviation.

Why not complete-case deletion?

- income alone has substantial non-response;
- BMI, cholesterol, smoking, and sleep also contribute missingness;
- deleting every row with any missing predictor can remove a large fraction of respondents and alter the analyzed sample.

Why an explicit category for categorical missingness?

It avoids inventing a substantive category such as the modal income group for a respondent who refused or did not report income. This choice is transparent and can later be tested in sensitivity analysis.



## 2.5 Train/Test Split

We use an **80/20 stratified split** with a fixed random seed.

- 80% training set: used for preprocessing fitting, cross-validation, model selection, and tuning.
- 20% test set: untouched until final evaluation.
- stratification preserves the diabetes/non-diabetes class ratio.

This split is respondent-level because the public BRFSS file contains cross-sectional survey records rather than longitudinal patient trajectories.



In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

RANDOM_STATE = 42
TEST_SIZE = 0.20

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame([
    {
        "split": "full",
        "n": len(y),
        "diabetes_n": int(y.sum()),
        "no_diabetes_n": int((y == 0).sum()),
        "diabetes_pct": float(y.mean() * 100),
    },
    {
        "split": "train",
        "n": len(y_train),
        "diabetes_n": int(y_train.sum()),
        "no_diabetes_n": int((y_train == 0).sum()),
        "diabetes_pct": float(y_train.mean() * 100),
    },
    {
        "split": "test",
        "n": len(y_test),
        "diabetes_n": int(y_test.sum()),
        "no_diabetes_n": int((y_test == 0).sum()),
        "diabetes_pct": float(y_test.mean() * 100),
    },
])

display(split_summary)
split_summary.to_csv("/kaggle/working/split_summary.csv", index=False)

assert set(X_train.index).isdisjoint(set(X_test.index))
assert len(X_train) + len(X_test) == len(X)


## 2.6 Leakage-Safe Preprocessing Pipeline

A single reusable `ColumnTransformer` is defined for the baseline model comparison.

Categorical variables are imputed with an explicit missing category and one-hot encoded. Numerical variables are median-imputed and standardized.

The transformer is **fit on `X_train` only**. Calling `fit_transform` on the full dataset before the split would leak information from the test set.



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1.0)),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ],
    remainder="drop",
)

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Training matrix:", X_train_ready.shape)
print("Test matrix:", X_test_ready.shape)
print("Transformed feature count:", len(feature_names))
print("First transformed feature names:")
print(feature_names[:25])

assert X_train_ready.shape[0] == len(X_train)
assert X_test_ready.shape[0] == len(X_test)
assert X_train_ready.shape[1] == X_test_ready.shape[1]


## 2.7 Stratified 5-Fold Cross-Validation

Model comparison will use **Stratified 5-Fold CV inside the training set**.

The preprocessing transformer will later be placed inside each model pipeline, so every CV fold fits its own imputation, encoding, and scaling using only that fold's training partition.

This prevents validation-fold information from influencing preprocessing.



In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

fold_rows = []
for fold, (fit_idx, val_idx) in enumerate(cv.split(X_train, y_train), start=1):
    y_fit = y_train.iloc[fit_idx]
    y_val = y_train.iloc[val_idx]
    fold_rows.append({
        "fold": fold,
        "fit_n": len(fit_idx),
        "validation_n": len(val_idx),
        "fit_diabetes_pct": float(y_fit.mean() * 100),
        "validation_diabetes_pct": float(y_val.mean() * 100),
    })

cv_fold_summary = pd.DataFrame(fold_rows)
display(cv_fold_summary)
cv_fold_summary.to_csv("/kaggle/working/cv_fold_summary.csv", index=False)


## 2.8 Frozen Data-Preparation Design

The following decisions are now frozen for the primary benchmark:

- **Target:** `DIABETE4=1` vs `DIABETE4=3`.
- **Predictors:** the 20 literature-grounded variables from Part I.
- **Categorical missingness:** explicit missing category.
- **Continuous missingness:** median imputation.
- **Encoding:** one-hot encoding with unknown-category handling.
- **Scaling:** standardize continuous predictors.
- **Split:** 80/20 stratified holdout, `random_state=42`.
- **Cross-validation:** stratified 5-fold CV within training data only.
- **Resampling:** none in the primary baseline. Any SMOTE/undersampling experiment must occur inside training/CV folds only.
- **Test set:** untouched until the final selected models are evaluated.

Part III can therefore compare algorithms on the **same cohort, same split, and same preprocessing logic**, making differences in performance attributable primarily to the algorithms rather than to inconsistent data preparation.



In [ ]:
preprocessing_summary = {
    "target_definition": "DIABETE4 1 vs 3",
    "n_primary_cohort": int(len(y)),
    "n_diabetes": int(y.sum()),
    "sample_diabetes_pct": round(float(y.mean() * 100), 4),
    "n_predictors_raw": len(PRIMARY_FEATURES),
    "complete_case_n": complete_case_n,
    "complete_case_pct": round(float(complete_case_pct), 4),
    "train_n": int(len(y_train)),
    "test_n": int(len(y_test)),
    "train_diabetes_pct": round(float(y_train.mean() * 100), 4),
    "test_diabetes_pct": round(float(y_test.mean() * 100), 4),
    "transformed_feature_count": int(len(feature_names)),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "cv_folds": 5,
    "categorical_missing_strategy": "constant -1 category",
    "numeric_missing_strategy": "median fit on training only",
    "categorical_encoding": "one-hot, handle_unknown=ignore",
    "numeric_scaling": "StandardScaler fit on training only",
}

with open("/kaggle/working/preprocessing_summary.json", "w", encoding="utf-8") as f:
    json.dump(preprocessing_summary, f, indent=2)

print(json.dumps(preprocessing_summary, indent=2))
